# Khảo sát dữ liệu gốc

Xem cấu trúc, dòng mẫu, dữ liệu thiếu và khoảng thời gian của `kz.csv` bằng PySpark.
Nguồn và mô tả các trường nằm trong `docs/data_source.md`.

## 1. Xem tiêu đề và vài bản ghi đầu

Xem ba bản ghi gốc với tên cột từ tiêu đề CSV. Mỗi bản ghi nằm trên một hàng.

In [1]:
from pathlib import Path
import sys

from IPython.display import display
from pyspark.sql import functions as F

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

SOURCE = ROOT / "data/raw/kz.csv"
REPORTS = ROOT / "reports"

from src.prepare_data import COLUMNS, read_source, start_spark

spark = start_spark()
raw = read_source(spark, SOURCE).cache()

spark.conf.set("spark.sql.repl.eagerEval.enabled", "true")
spark.conf.set("spark.sql.repl.eagerEval.truncate", "0")

sample = raw.limit(3).orderBy("source_record")
display(sample.select(*[
    F.col("fields")[index].alias(name)
    for index, name in enumerate(COLUMNS)
]))

event_time,order_id,product_id,category_id,category_code,brand,price,user_id
2020-04-24 11:50:39 UTC,2294359932054536986,1515966223509089906,2268105426648170900,electronics.tablet,samsung,162.01,1515915625441993984
2020-04-24 11:50:39 UTC,2294359932054536986,1515966223509089906,2268105426648170900,electronics.tablet,samsung,162.01,1515915625441993984
2020-04-24 14:37:43 UTC,2294444024058086220,2273948319057183658,2268105430162997728,electronics.audio.headphone,huawei,77.52,1515915625447879434


## 2. Đếm số trường trên toàn file

Ba bản ghi đầu chỉ là mẫu. Đếm số trường trên từng bản ghi để biết có bao nhiêu dòng đủ 8 trường như tiêu đề và bao nhiêu dòng có cấu trúc khác.

In [2]:
structure = raw.groupBy("field_count").agg(F.count("*").alias("rows")).orderBy("field_count")
width_counts = structure.collect()
display(structure)
print(f"Source records: {sum(row.rows for row in width_counts):,}")

field_count,rows
6,431954
8,2201567


Source records: 2,633,521


## 3. Xem mẫu dòng có 6 trường

Xem hai bản ghi gốc, một dòng có giá trị cuối và một dòng để trống.
Tên cột `field_1` đến `field_6` chỉ vị trí trong bản ghi gốc.

In [3]:
full_rows = raw.filter("field_count = 8")
short_rows = raw.filter("field_count = 6")

with_last_value = short_rows.filter(F.col("fields")[5] != "").orderBy("source_record").limit(1)
without_last_value = short_rows.filter(F.col("fields")[5] == "").orderBy("source_record").limit(1)
samples = with_last_value.unionByName(without_last_value).orderBy("source_record")

display(samples.select("source_record", *[
    F.col("fields")[index].alias(f"field_{index + 1}")
    for index in range(6)
]))

source_record,field_1,field_2,field_3,field_4,field_5,field_6
1813,2020-05-02 11:12:20 UTC,2300138854784434309,1515966223509353669,,0.02,1515915625453675584
28819,1970-01-01 00:33:40 UTC,2340102743319904671,1515966223523303300,,13.87,


## 4. Đo dữ liệu thiếu trong nhóm đủ 8 trường

Chỉ nhóm đủ 8 trường có thể đếm ô thiếu theo tên cột gốc. Mẫu số của bảng là số dòng trong nhóm này.

In [4]:
import csv

from src.prepare_data import missing_counts

full_width = full_rows.select(*[
    F.when(F.trim(F.col("fields")[index]) != "", F.col("fields")[index]).alias(name)
    for index, name in enumerate(COLUMNS)
])
full_count = full_rows.count()
missing = missing_counts(full_width, COLUMNS)
quality = [
    (name, full_count, missing[name], round(100 * missing[name] / full_count, 4))
    for name in COLUMNS
]
header = ["column", "full_width_rows", "missing_count", "missing_pct"]
display(spark.createDataFrame(quality, header))

REPORTS.mkdir(parents=True, exist_ok=True)
with (REPORTS / "row_structure.csv").open("w", encoding="utf-8-sig", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(["field_count", "rows"])
    writer.writerows(width_counts)
with (REPORTS / "column_quality.csv").open("w", encoding="utf-8-sig", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(header)
    writer.writerows(quality)

column,full_width_rows,missing_count,missing_pct
event_time,2201567,0,0.0
order_id,2201567,0,0.0
product_id,2201567,0,0.0
category_id,2201567,0,0.0
category_code,2201567,612202,27.8076
brand,2201567,112670,5.1177
price,2201567,0,0.0
user_id,2201567,1637398,74.3742


## 5. Quan sát thời gian mua

Đọc giá trị đầu mỗi dòng thành thời điểm UTC để thống kê khoảng ngày theo năm.
Dòng có năm trống là dòng không đọc được thời điểm.

In [5]:
try:
    dates = raw.select(F.try_to_timestamp(F.col("fields")[0], F.lit("yyyy-MM-dd HH:mm:ss 'UTC'")).alias("event_time"))
    by_year = dates.groupBy(F.year("event_time").alias("year")).agg(
        F.count("*").alias("rows"),
        F.min("event_time").alias("first_event"),
        F.max("event_time").alias("last_event"),
    ).orderBy("year")
    display(by_year)
finally:
    raw.unpersist()
    spark.stop()

year,rows,first_event,last_event
1970,19631,1970-01-01 00:33:40,1970-01-01 00:33:40
2020,2613890,2020-01-05 03:26:46,2020-11-21 10:10:30
